# Breast Cancer Transcriptomics and Machine Learning Pipeline

### Computational Framework for Molecular Subtype Classification and Biomarker Discovery

This study presents an end-to-end bioinformatics and machine learning pipeline for classifying breast cancer tumor samples into their clinical molecular subtypes: **Basal-like, HER2-enriched, Luminal A, Luminal B, and Normal-like** using patient microarray profiles (**GSE45827**).

---

## Dataset Characteristics

The dataset is sourced from the curated **CuMiDa (Curated Microarray Database)** repository, which provides extensively profiled and clinically verified gene expression matrices:

* **Platform:** Affymetrix Human Genome U133 Plus 2.0 Array
* **Initial Features:** 54,675 probe signal intensities (housekeeping features filtered out using a Variance Threshold of 0.1)
* **Clinical Cohort:** 137 tumor samples (14 laboratory cell line controls were removed to prevent confounding biological signals)

### Clinical Subtype Cohort Distribution

| Molecular Subtype | Sample Count (N=137) | Cohort % | Clinical Profile and Therapeutic Targets |
| :--- | :---: | :---: | :--- |
| **Basal-like** | 41 | 29.9% | Aggressive, highly proliferative, triple-negative breast cancer (TNBC) lacking ER, PR, and HER2 receptors. |
| **HER2-enriched** | 30 | 21.9% | Driven by amplification of the ERBB2 (HER2) receptor; responsive to HER2-targeted agents (Trastuzumab). |
| **Luminal A** | 29 | 21.2% | Estrogen receptor-positive (ER+), slow-growing, low proliferation index. Favorable clinical prognosis. |
| **Luminal B** | 30 | 21.9% | Estrogen receptor-positive (ER+), highly proliferative and clinically aggressive. |
| **Normal-like** | 7 | 5.1% | Adjacent non-tumor normal tissue controls serving as our biological baseline. |

---

## Pipeline Architecture (Methodologically Audited)

Following a rigorous formal methodology audit to prevent data leakage and guarantee clinical translational validity, the analytical workflow progresses through six primary phases:

1. **Data Preprocessing & Cohort Splitting:** Immediate 80/20 stratification into a Discovery Cohort and a strict Holdout Cohort prior to any downstream normalization.
2. **Exploratory Data Analysis (Discovery Cohort Only):** Global Quantile Normalization applied exclusively to the Discovery Cohort for latent space visualizations (PCA, t-SNE, UMAP) and Differential Gene Expression (DGE) analysis.
3. **Rigorous Machine Learning Validation (Nested CV):** Construction of a `scikit-learn` Pipeline wrapping fold-local Quantile Normalization, Ensemble Feature Selection, and Classifiers, evaluated via `RepeatedStratifiedKFold` CV.
4. **Holdout Validation & Bootstrap Stability:** Final model evaluated on the untouched 20% Holdout Cohort. Implementation of 1000-iteration Bootstrapping to calculate 95% Confidence Intervals.
5. **Clinical Utility Metrics:** Generation of Calibration Curves (Reliability Diagrams), Brier Scores, and Decision Curve Analysis (DCA).
6. **Clinical Explainability & Pathway Enrichment:** Computation of SHAP values and Gene Ontology (GO) pathway enrichment on the signature biomarkers.


## Section 0: Environment Setup and Imports
We begin by importing all necessary Python packages, setting up the file path architecture, and defining global visualization aesthetics.

In [ ]:
import sys
import os
import warnings
from pathlib import Path

# Scientific computing & data wrangling
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

# Machine Learning & Unsupervised Clustering
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_validate, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    roc_curve, auc, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.utils import resample

# Bioinformatic APIs, Explainability & Pathway Enrichment
import shap
import mygene
import gseapy as gp
import umap

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Setup absolute file path architecture
PROJECT_ROOT = Path("..").resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "data" / "artifacts"

# Create directories if they do not exist
for d in [PROCESSED_DATA_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Aesthetic configuration
sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
warnings.filterwarnings("ignore")

print("Environment ready! Imports complete.")


## Section 1: Data Loading and Strict Cohort Splitting

Gene expression microarray data contains measurements of thousands of RNA transcripts across tumor samples. The dataset (**GSE45827**, curated and made accessible by the CuMiDa database Feltes et al., 2019) contains 151 samples and 54,675 probes. 
However, **21 of these samples are cell lines**, which represent laboratory cell cultures. Cell lines undergo genetic drift and lack the microenvironment of real patient tumors, representing potential confounders. 
In this section, we load the raw data, remove the cell line samples, and immediately split the remaining cohort into an **80% Discovery Cohort** and a **20% Holdout Cohort** to prevent data leakage during subsequent normalization and feature selection.

In [ ]:
# ── 1.1 Load Raw CSV Data ──
csv_path = RAW_DATA_DIR / "Breast_GSE45827.csv"
df_raw = pd.read_csv(csv_path)

print(f"Raw dataset shape: {df_raw.shape[0]} samples x {df_raw.shape[1]:,} features")

# ── 1.2 Remove Cell Line Samples & ID Column ──
df = df_raw[df_raw['type'] != 'cell_line'].copy()
df = df.drop(columns=['samples'])
print(f"Removed {len(df_raw) - len(df)} cell line samples.")
print(f"Remaining clinical samples shape: {df.shape[0]} samples x {df.shape[1]:,} features")

# ── 1.3 Optimize Memory via Downcasting ──
feat_cols = df.columns.drop("type")
df[feat_cols] = df[feat_cols].astype(np.float32)

# Save parsed raw parquet
df.to_parquet(PROCESSED_DATA_DIR / "breast_cancer.parquet", index=False)

# ── 1.4 Strict 80/20 Cohort Splitting ──
X_raw = df[feat_cols].values
y_raw = df['type'].values

le = LabelEncoder()
y_enc = le.fit_transform(y_raw)

# Splitting BEFORE any normalization or feature selection
X_discover, X_holdout, y_discover, y_holdout = train_test_split(
    X_raw, y_enc, test_size=0.20, stratify=y_enc, random_state=42
)

print(f"\nData split successful:")
print(f"  Discovery Cohort: {X_discover.shape[0]} samples (used for EDA and CV)")
print(f"  Holdout Cohort  : {X_holdout.shape[0]} samples (locked away for final validation)")

import gc
del df_raw
gc.collect()


## Section 2: Exploratory Data Analysis (Discovery Cohort Only)

> [!WARNING]
> To strictly prevent data leakage, the exploratory visualizations (PCA, t-SNE, UMAP) in this section are performed **only on the 80% Discovery Cohort**. For the visualizations, we apply a global Quantile Normalization to this cohort. During the Machine Learning phase, normalization will be applied dynamically inside the cross-validation folds.

To standardize microarray signal intensities across samples and remove technical batch variations, we implement **Quantile Normalization (QN)** on the Discovery Cohort. This forces all samples to have the exact same distribution of intensities, adjusting for technical noise without distorting biological signals.

In [ ]:
# ── 2.1 Global Quantile Normalization on Discovery Cohort (for EDA) ──
# Sort columns (samples) to find rank order
X_sorted = np.sort(X_discover, axis=1)
reference = X_sorted.mean(axis=0)  # average profile

ranks = np.argsort(np.argsort(X_discover, axis=1), axis=1)  # indices of sorted values
X_discover_qn = reference[ranks]            # project ranks onto average profile

# ── 2.2 Dimensionality Reduction (PCA, t-SNE, UMAP) ──
# Filter to top 5000 variable genes for speed and noise reduction in EDA
gene_vars = np.var(X_discover_qn, axis=0)
top5k_idx = np.argsort(gene_vars)[-5000:]
X_top5k = X_discover_qn[:, top5k_idx]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_top5k)

# PCA
pca = PCA(n_components=50, random_state=42)
X_pca50 = pca.fit_transform(X_scaled)

# t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_pca50)

# UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_pca50)

# ── 2.3 Visualize ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = le.classes_

for i, st in enumerate(class_names):
    mask = y_discover == i
    axes[0].scatter(X_pca50[mask, 0], X_pca50[mask, 1], label=st, alpha=0.8)
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=st, alpha=0.8)
    axes[2].scatter(X_umap[mask, 0], X_umap[mask, 1], label=st, alpha=0.8)

axes[0].set_title("PCA (PC1 vs PC2)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].set_title("t-SNE")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")

axes[2].set_title("UMAP")
axes[2].set_xlabel("UMAP 1")
axes[2].set_ylabel("UMAP 2")
axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.suptitle("Latent Space Projections of Breast Cancer Subtypes (Discovery Cohort)", fontsize=16)
plt.tight_layout()
plt.show()


### Dimensionality Reduction Observations
* **PCA:** Basal tumors completely isolate themselves on the far right along PC1, indicating a highly unique, transcriptionally extreme signature. Normal tissue forms a tight cluster distinct from tumors.
* **t-SNE & UMAP:** Non-linear mapping reveals a continuum that mirrors clinical progression: Normal -> Luminal A -> Luminal B -> HER2 -> Basal.

## Section 3: Rigorous Machine Learning Preprocessing (Pipelines)

> [!IMPORTANT]
> To prevent data leakage, **all** normalization and feature selection steps must be calculated uniquely for each training fold during cross-validation, and never on the validation fold or the 20% holdout.

We achieve this by creating custom `scikit-learn` `TransformerMixin` objects for Quantile Normalization and Ensemble Feature Selection, and wrapping them in a `Pipeline`.

In [ ]:
# ── 3.1 Fold-Local Quantile Normalizer ──
class QuantileNormalizer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.reference_distribution = None
        
    def fit(self, X, y=None):
        # Calculate reference distribution (mean of sorted columns) on the TRAINING fold
        X_sorted = np.sort(X, axis=1)
        self.reference_distribution = X_sorted.mean(axis=0)
        return self
        
    def transform(self, X):
        # Project ranks of the input data onto the learned reference distribution
        ranks = np.argsort(np.argsort(X, axis=1), axis=1)
        return self.reference_distribution[ranks]

# ── 3.2 Fold-Local Ensemble Feature Selector ──
class EnsembleFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k_features=1500):
        self.k_features = k_features
        self.support_mask_ = None
        
    def fit(self, X, y=None):
        # Calculate multiple selection metrics
        anova_sel = SelectKBest(f_classif, k=self.k_features).fit(X, y)
        
        # We can combine Multiple Selection methods here, but for computational 
        # efficiency in Repeated CV, we'll use strong ANOVA + RF Importance.
        rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        rf.fit(X, y)
        rf_importances = rf.feature_importances_
        rf_threshold = np.sort(rf_importances)[-self.k_features]
        rf_sel_mask = rf_importances >= rf_threshold
        
        # Consensus: Selected by either ANOVA or RF
        self.support_mask_ = anova_sel.get_support() | rf_sel_mask
        return self
        
    def transform(self, X):
        return X[:, self.support_mask_]
        
    def get_support(self):
        return self.support_mask_

print("Pipeline Transformers constructed successfully.")


## Section 4: Repeated Stratified Nested CV & Ablation Logging

We evaluate our pipeline configurations using **Repeated Stratified K-Fold** (5 folds, 3 repeats = 15 total fits per model) on the Discovery Cohort.
To prove the internal validity of our feature selection, we implement an **Ablation test**:
1. **Full Pipeline:** QN -> Ensemble Feature Selection -> Random Forest
2. **No Feature Selection:** QN -> Random Forest (to prove Biomarker utility)
3. **No Normalization:** Ensemble Feature Selection -> Random Forest (to prove QN utility)


In [ ]:
# Define Pipelines
pipelines = {
    'Full Pipeline (QN + FS + RF)': Pipeline([
        ('qn', QuantileNormalizer()),
        ('fs', EnsembleFeatureSelector(k_features=1000)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    'No Feature Selection (QN + RF)': Pipeline([
        ('qn', QuantileNormalizer()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    'No Normalization (FS + RF)': Pipeline([
        ('fs', EnsembleFeatureSelector(k_features=1000)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
}

# Repeated Stratified K-Fold
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

ablation_results = []

print("Running Repeated Stratified CV (this may take a minute)...")
for name, pipe in pipelines.items():
    print(f"Evaluating: {name}")
    cv_res = cross_validate(pipe, X_discover, y_discover, cv=rskf, scoring='f1_weighted', n_jobs=-1)
    ablation_results.append({
        'Pipeline Configuration': name,
        'Mean CV Weighted F1': cv_res['test_score'].mean(),
        'Std CV Weighted F1': cv_res['test_score'].std()
    })

ablation_df = pd.DataFrame(ablation_results).sort_values(by='Mean CV Weighted F1', ascending=False)
display(ablation_df)


## Section 5: External Validation (Holdout) & Clinical Utility

Having validated our architecture internally, we now fit the **Full Pipeline** on the entire Discovery Cohort and evaluate it on the strictly segregated 20% Holdout Cohort.

### 5.1 Bootstrapping Confidence Intervals
To quantify uncertainty and address potential minority class instability, we use 1000 bootstrap iterations on the holdout predictions to compute 95% Confidence Intervals.

In [ ]:
# Train final model
final_pipe = pipelines['Full Pipeline (QN + FS + RF)']
final_pipe.fit(X_discover, y_discover)

# Predict on holdout
y_pred_holdout = final_pipe.predict(X_holdout)

# Bootstrapping for 95% CIs
n_iterations = 1000
bootstrapped_scores = []

for i in range(n_iterations):
    # Resample indices with replacement
    indices = resample(np.arange(len(y_holdout)), random_state=i)
    if len(np.unique(y_holdout[indices])) < 2:
        continue
    
    score = f1_score(y_holdout[indices], y_pred_holdout[indices], average='weighted')
    bootstrapped_scores.append(score)

mean_score = np.mean(bootstrapped_scores)
ci_lower = np.percentile(bootstrapped_scores, 2.5)
ci_upper = np.percentile(bootstrapped_scores, 97.5)

print(f"Holdout Weighted F1 Score: {f1_score(y_holdout, y_pred_holdout, average='weighted'):.4f}")
print(f"95% Confidence Interval (Bootstrapped): [{ci_lower:.4f}, {ci_upper:.4f}]")

plt.figure(figsize=(8, 4))
sns.histplot(bootstrapped_scores, bins=30, kde=True, color='teal')
plt.axvline(mean_score, color='red', linestyle='dashed', linewidth=2, label=f'Mean: {mean_score:.3f}')
plt.axvline(ci_lower, color='black', linestyle='dotted', linewidth=2, label='95% CI')
plt.axvline(ci_upper, color='black', linestyle='dotted', linewidth=2)
plt.title("Bootstrap Distribution of Weighted F1 Score (Holdout Cohort)")
plt.xlabel("Weighted F1 Score")
plt.legend()
plt.show()


### 5.2 Probabilistic Calibration & Brier Score
Calibration curves measure if the predicted probabilities align with true empirical frequencies. We isolate the most aggressive class (Basal-like) for the calibration demonstration.

In [ ]:
# Get probabilities
y_prob_holdout = final_pipe.predict_proba(X_holdout)

# Focus on 'Basal-like' subtype
basal_idx = list(class_names).index('Basal')
y_holdout_basal = (y_holdout == basal_idx).astype(int)
y_prob_basal = y_prob_holdout[:, basal_idx]

prob_true, prob_pred = calibration_curve(y_holdout_basal, y_prob_basal, n_bins=5)
brier = brier_score_loss(y_holdout_basal, y_prob_basal)

plt.figure(figsize=(7, 7))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label=f'RF Pipeline (Brier={brier:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')
plt.title("Calibration Curve (Reliability Diagram) for Basal Subtype")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.legend()
plt.show()


### 5.3 Decision Curve Analysis (DCA)
DCA evaluates the clinical utility of the model by calculating the Net Benefit across various risk thresholds compared to default strategies (Treat All vs Treat None).

In [ ]:
def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefits = []
    n = len(y_true)
    for pt in thresholds:
        tp = np.sum((y_prob >= pt) & (y_true == 1))
        fp = np.sum((y_prob >= pt) & (y_true == 0))
        if pt == 1.0: pt = 0.9999
        net_benefit = (tp / n) - (fp / n) * (pt / (1 - pt))
        net_benefits.append(net_benefit)
    return np.array(net_benefits)

thresholds = np.linspace(0.01, 0.99, 100)
nb_model = calculate_net_benefit(y_holdout_basal, y_prob_basal, thresholds)
nb_all = calculate_net_benefit(y_holdout_basal, np.ones_like(y_prob_basal), thresholds)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, nb_model, label="Basal Prediction Model", linewidth=2, color='blue')
plt.plot(thresholds, nb_all, label="Treat All", linestyle='--', color='gray')
plt.plot(thresholds, np.zeros_like(thresholds), label="Treat None", linestyle='-', color='black')
plt.ylim([-0.05, 0.4])
plt.title("Decision Curve Analysis (DCA) - Basal Subtype")
plt.xlabel("Threshold Probability")
plt.ylabel("Net Benefit")
plt.legend()
plt.show()


## Section 6: Biological Interpretation (Explainable AI & Pathways)

To provide biological interpretability and strengthen mechanistic insights, we extract the optimal feature subset selected by our pipeline, generate SHAP attributions, and run Gene Ontology (GO) enrichment via `gseapy`.

In [ ]:
# Extract selected genes from the fitted pipeline
selector = final_pipe.named_steps['fs']
mask = selector.get_support()
selected_genes = feat_cols[mask].tolist()

print(f"Extracted {len(selected_genes)} consensus biomarker genes from the pipeline.")

# Train a standalone Explainer Model on the selected features to generate SHAP values
X_discover_qn = final_pipe.named_steps['qn'].transform(X_discover)
X_discover_sel = final_pipe.named_steps['fs'].transform(X_discover_qn)

explainer_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
explainer_rf.fit(X_discover_sel, y_discover)

# Compute TreeSHAP values
explainer = shap.TreeExplainer(explainer_rf)
shap_values = explainer.shap_values(X_discover_sel)

# Plot Global Feature Importance (Top 15 Genes)
shap.summary_plot(shap_values, X_discover_sel, feature_names=selected_genes, max_display=15, class_names=class_names)


### Gene Ontology (GO) Biological Process Enrichment
Mapping Affymetrix probe identifiers to official HGNC symbols and running functional enrichment.

In [ ]:
# Map Affymetrix Probes to Gene Symbols using mygene
mg = mygene.MyGeneInfo()
gene_info = mg.querymany(selected_genes[:1000], scopes='reporter', fields='symbol', species='human', as_dataframe=True)

# Filter valid symbols
if 'symbol' in gene_info.columns:
    symbols = gene_info['symbol'].dropna().unique().tolist()
    print(f"Mapped to {len(symbols)} unique gene symbols. Running Enrichr...")

    # Run Pathway Enrichment
    try:
        enrich_res = gp.enrichr(
            gene_list=symbols,
            gene_sets='GO_Biological_Process_2021',
            organism='human',
            outdir=None
        )
        display(enrich_res.results.head(10)[['Term', 'Adjusted P-value', 'Overlap', 'Genes']])
    except Exception as e:
        print("Enrichr API request failed or timed out. Please check network connection.")
else:
    print("Could not map probe IDs to symbols.")


## Section 7: Methodological Limitations and Future Directions

In accordance with rigorous academic standards, we explicitly acknowledge the following limitations of this study and propose directions for future work to address remaining Literature Gap Analysis rankings:

1. **Survival and Therapy-Response Endpoints:** This dataset (`GSE45827`) provides transcriptomic profiles and clinical subtypes, but lacks longitudinal patient tracking. Future studies must integrate Overall Survival (OS), Disease-Free Survival (DFS), and therapy-response metadata to build true prognostic and predictive utility models using Cox Proportional Hazards or DeepSurv algorithms.
2. **Batch Effect Benchmarking:** The current dataset is highly curated and lacks technical batch tracking labels (e.g., scanning date, hospital origin). Robust clinical translation requires benchmarking the pipeline against multi-institutional cohorts and applying batch-correction algorithms like ComBat.
3. **External Cohort Validation:** While we used a strict 20% holdout cohort to proxy external validation, true external validity requires testing the locked pipeline on an entirely independent dataset from a different platform (e.g., RNA-Seq) or institution.
4. **Multi-Omics & Single-Cell Resolution:** Bulk microarray data captures average expression. Moving forward, incorporating scRNA-seq and spatial transcriptomics will untangle the tumor microenvironment (TME) heterogeneity and isolate the specific immune and stromal cell contributions to these subtypes.
5. **Advanced Architectures & Foundation Models:** The transition from traditional ML to Graph Neural Networks (GNNs) utilizing Protein-Protein Interaction (PPI) networks, Causal Inference layers, or fine-tuned Foundation Models (like Geneformer or scGPT) represents the next evolutionary step for this pipeline.
